# 第8章 MCP (Model Context Protocol) 連携

外部の **MCP サーバ** が提供するツールを、LangGraph エージェントから呼び出します。

**所要時間**: 60〜90分

## 学ぶこと
1. MCP の基本構成(Server / Client / Transport)
2. 自作 MCP サーバ(Python `FastMCP`)を立てて接続
3. 公式 filesystem MCP サーバを `npx` 経由で接続
4. **複数の MCP サーバを 1 つのエージェントに統合**

## MCP とは

**Model Context Protocol (MCP)** は Anthropic が 2024 年に提唱した、LLM とツール・データソースをつなぐためのオープン標準プロトコルです。

```
  ┌───────────────┐                  ┌─────────────────┐
  │ MCP Client    │  ←── JSON-RPC ──→│ MCP Server      │
  │ (=ホストアプリ)│       (Transport) │ (=ツール提供者)  │
  │ 例: LangChain │                  │ 例: filesystem  │
  │     Claude    │                  │     GitHub      │
  │     Cursor    │                  │     Slack       │
  └───────────────┘                  └─────────────────┘
```

**ポイント**
- サーバはツール群を「メニュー」として公開する(`list_tools` / `call_tool` 等の標準コマンド)
- クライアントは「どのサーバに何があるか」を統一プロトコルで問い合わせ、呼び出す
- 既製の MCP サーバ(filesystem、GitHub、PostgreSQL、Slack 等)が公開されており、自前で実装しなくても接続するだけで使える
- 自作も簡単で、Python なら `FastMCP` クラスで関数を `@tool()` デコレートするだけ

**LangChain / LangGraph との関係**
- `langchain-mcp-adapters` パッケージが、MCP サーバのツールを **LangChain Tool に変換** してくれる
- 変換後のツールは、これまで習った `create_agent` 等にそのまま渡せる

**Transport の種類**
- **stdio**: クライアントがサーバプロセスを subprocess で起動し、標準入出力で通信。本章で使う方式
- **HTTP/SSE**: サーバ常駐型。リモートやマルチユーザー用途(本章では割愛)

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

# .env を読み込む(これまでの章と同じ)。LangSmith のトレースもここで有効になる
load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

In [ ]:
# Node.js が DevContainer に入っているか確認(filesystem MCP で必要)
import shutil
import subprocess

if shutil.which("npx"):
    out = subprocess.run(["node", "--version"], capture_output=True, text=True)
    print(f"Node.js OK: {out.stdout.strip()}")
else:
    print("⚠️ npx が見つかりません。DevContainer の Rebuild が必要かもしれません。")

In [ ]:
from langchain_aws import ChatBedrockConverse

# MCP ツールを呼び出すエージェントの頭脳となる LLM(これまでと同じ)
llm = ChatBedrockConverse(model=CHAT_MODEL_ID, region_name=AWS_REGION, temperature=0)

## 1. パート1: 自作 MCP サーバ(math)に接続する

`mcp_servers/math_server.py` に最小の MCP サーバを用意しています。中身を確認してみましょう。

In [ ]:
# サーバ実装を表示する
with open("mcp_servers/math_server.py", encoding="utf-8") as f:
    print(f.read())

ポイント:
- `FastMCP("math")` で MCP サーバインスタンスを作成("math" はサーバ名)
- `@mcp.tool()` で公開したい関数をデコレート。docstring がツール説明、型ヒントがスキーマになる(`@tool` の MCP 版と考えれば OK)
- `mcp.run(transport="stdio")` で標準入出力経由のサーバを起動

クライアントはこのファイルを `python math_server.py` で **subprocess として起動** し、stdio で対話します。

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP サーバへの接続設定を辞書で渡す
# キーがクライアント側でのサーバ識別名、値が起動方法
client = MultiServerMCPClient({
    "math": {
        "command": "python",                       # 起動コマンド
        "args": ["mcp_servers/math_server.py"],    # 引数
        "transport": "stdio",                       # stdio で接続
    }
})

# サーバを起動して公開されているツール一覧を取得
# (この関数は async。Jupyter なら top-level await が使える)
math_tools = await client.get_tools()

# 取得できたツールを確認。MCP のツールも LangChain の Tool オブジェクトとして扱える
for t in math_tools:
    print(f"- {t.name}: {t.description}")

In [ ]:
from langchain.agents import create_agent

# MCP 経由で取得したツールを、第3章・第5章で使った create_agent にそのまま渡す。
# (create_agent は langchain 1.x 標準。旧 create_react_agent の後継)
math_agent = create_agent(llm, tools=math_tools)

# 計算を依頼。MCP サーバの add/multiply が呼ばれて結果が返ってくるはず
result = await math_agent.ainvoke({
    "messages": [{"role": "user", "content": "3.14 と 2.86 を足してから、その結果に 10 を掛けて。"}]
})

for m in result["messages"]:
    m.pretty_print()

ツール呼び出しのログに `add` と `multiply` が現れていれば、MCP サーバ経由でツールが実行されている証拠です。

## 2. パート2: 公式 filesystem MCP サーバを追加

**Anthropic が公開している** ファイルシステム MCP サーバを追加します。
Node 製で、`npx -y @modelcontextprotocol/server-filesystem <許可ディレクトリ>` で起動できます。

**セキュリティ**: filesystem MCP は明示的に許可したディレクトリ配下しかアクセスしません。
本章では `./data/` 配下だけを公開します。

In [ ]:
# 公開対象ディレクトリの絶対パスを取得(filesystem MCP は絶対パスを要求する)
data_dir = os.path.abspath("./data")
print("公開ディレクトリ:", data_dir)

# 中身を確認
for f in sorted(os.listdir(data_dir)):
    print(" ", f)

In [ ]:
# math + filesystem の 2 つのサーバを同時に登録
multi_client = MultiServerMCPClient({
    "math": {
        "command": "python",
        "args": ["mcp_servers/math_server.py"],
        "transport": "stdio",
    },
    "filesystem": {
        "command": "npx",
        # -y: 確認プロンプトを自動的に yes。 -y がないと初回 npx で待ち状態になりタイムアウトする
        # 引数の最後でアクセス許可ディレクトリを指定する
        "args": ["-y", "@modelcontextprotocol/server-filesystem", data_dir],
        "transport": "stdio",
    },
})

# 両方のサーバを起動してツールを集約
# 初回 npx 実行はパッケージダウンロードのため 30 秒〜1 分程度かかることがある
all_tools = await multi_client.get_tools()

print(f"取得ツール総数: {len(all_tools)}\n")
for t in all_tools:
    # filesystem の公式サーバは list_directory, read_file, write_file など複数のツールを公開する
    print(f"- {t.name}")

In [ ]:
# 両方のツールを持ったエージェントを作成
multi_agent = create_agent(llm, tools=all_tools)

# ファイル一覧 + 内容を読んで要約する依頼
result = await multi_agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            f"{data_dir} 配下にあるテキストファイルを一覧して、"
            "それぞれの内容を 2 行程度に要約してください。"
        )
    }]
})

for m in result["messages"]:
    m.pretty_print()

verbose に出ているツール呼び出しを見ると、`list_directory` → `read_text_file` → LLM が要約、という流れが追えるはずです。
LangSmith のダッシュボードで同じトレースを見ると、各ツール呼び出しの引数と戻り値も確認できます。

## 3. math と filesystem を組み合わせるタスク

せっかく両サーバが繋がっているので、両方のツールを跨ぐタスクを依頼してみます。

In [ ]:
# math(自作)と filesystem(公式)の両方のツールを跨ぐ複合タスク。
# 「ファイルを読む(filesystem の read_file)」→「文字数を数えて合計する(math の add)」という流れを、
# エージェントが自分で判断し、複数サーバのツールを組み合わせて解くのが見どころ。
result = await multi_agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            f"{data_dir} の各ファイルの文字数を取得して、その合計を計算してください。"
            "文字数の取得は read_file の結果の長さで判断してOKです。"
            "最終的に各ファイル名と文字数、そして合計を報告してください。"
        )
    }]
})
print("=== 最終回答 ===")
print(result["messages"][-1].content)

## まとめ

- **MCP** は LLM とツールをつなぐオープン標準。クライアント/サーバ/トランスポートに分かれる
- 自作サーバは `FastMCP` + `@mcp.tool()` で数行で書ける
- 公式の既製サーバ(filesystem 等)を取り込むだけで、エージェントの能力が一気に広がる
- `langchain-mcp-adapters` の `MultiServerMCPClient` で **複数サーバを 1 つのエージェントに統合** できる
- 取得した MCP ツールは LangChain Tool として扱えるので、`create_agent` などの既習 API にそのまま渡せる

### 発展トピック(本章スコープ外)

- **HTTP / SSE transport**: サーバ常駐型、リモート MCP、マルチユーザー
- **認可 / トークン**: 本格運用ではアクセス制御を MCP サーバ側で実装する
- **既製の MCP サーバ群**:
  - `@modelcontextprotocol/server-github`: GitHub Issues / PRs / コード閲覧
  - `@modelcontextprotocol/server-postgres`: SQL 実行
  - `@modelcontextprotocol/server-slack`: メッセージ送受信
  - その他多数: https://github.com/modelcontextprotocol/servers
- **AWS 提供の MCP サーバ**: Bedrock Knowledge Bases や Lambda、Athena への接続用 MCP サーバが AWS 公式から提供されている
- **マルチエージェント × MCP**: 07 章の Supervisor 構成と組み合わせて、各エージェントが異なる MCP サーバ群を使うように拡張できる

次は [09 総まとめ](../09_wrapup/) で、ハンズオン全体を振り返ります。